# Model predicting thermal sensation using given database

Link to database: https://github.com/CenterForTheBuiltEnvironment/ashrae-db-II.git

Packages: 
1. pandas
2. scipy 
3. math - no need 
4. numpy
5. scikit
6. tensorflow and keras

## Data preprocessing

### Handling NaN values

Given the fact that the dataset consists of a collection of different studies, each of which take into consideration varied parameters, the following code calculates the amount of NaN values on each column of the dataframe. The aim here is to find the parameters most commonly used in studies to create a final dataset as consistent as possible.

In [1]:
def get_nan_values(df):

    #create percentages
    size = df['index'].size 
    nan_array = df.isnull().sum() / size * 100 #creates a series of the percentages

    #store in file
    nan_array_sorted = nan_array.sort_values(ascending=True) #sorts throught the series 
    nan_array_sorted = nan_array_sorted[nan_array_sorted<50.0] #only keeps columns with below 50% NaN cells 

    out_path = str(pathlib.Path().resolve() / "outputs") + '\data_sorted.csv'
    nan_array_sorted.to_csv(out_path, header = None, sep = ' ')

def map_non_numeric_features(df): 
    
    #turn thermal preference into numbers
    mapping = {
        "cooler": -1,
        "warmer": 1,
        "no change": 0
    }

    df["thermal_preference"] = df["thermal_preference"].map(mapping)

    #turn season into numbers 
    mapping = { 
        "summer": -1,
        "winter": 1,
    }

    df["season"] = df["season"].map(mapping)

    return df

def remove_nan_values(df): 

    size = df['index'].size 
    df_main  = df[['index','ta', 'rh', 'vel', 'met', 'clo', 'tr', 'thermal_preference', 'season', 'age', 'pmv', 'thermal_sensation']]

    #removing NaN values
    df_main = df_main.dropna()
    size_new = df_main['index'].size 
    loss = 100 - ( (size_new / size) * 100)
    log_message(msg = f"By removing NaN values, with starting data size being: {size} and new size being: {size_new} the data loss is: {loss} %", path = '/losses_log.txt')

    #making databases to use for outlier detection
    df_outliers = df_main[['index','ta', 'rh', 'vel', 'met', 'clo', 'tr', 'thermal_preference', 'season', 'age', 'pmv', 'thermal_sensation']]
    df_measures = df_main[['ta', 'rh', 'vel', 'met', 'clo', 'tr', 'thermal_preference', 'season', 'age', 'pmv']]

    #make correlation matrix and store
    corr = df_measures.corr()
    out_path = str(pathlib.Path().resolve() / "outputs") + '\correlations.csv'
    corr.to_csv(out_path)

    return df_outliers, df_measures, loss

def nan_handling(df): 

    #find and sort percentages of NaN values per column
    get_nan_values(df)

    #after analyzing file, turn relevant non numeric columns to numeric 
    df_numbers = map_non_numeric_features(df)

    #remove NaN rows and keep only relevant columns after analysis
    df_outliers, df_measures, loss = remove_nan_values(df_numbers)

    return df_outliers, df_measures, loss

### Outlier detection

For the outlier detection different methods are tried below. 

In [2]:
def data_plots(df_outliers): 

    out_path = str(pathlib.Path().resolve() / 'outputs')
    for col in df_outliers.columns:
        #scatter plot to visualize the data accurately
        fig, ax = plt.subplots(figsize=(16,8))
        ax.scatter(df_outliers[col], df_outliers['index'])
        ax.set_xlabel(col)
        ax.set_ylabel('Index')
        ax.set_title(f"{col} vs Index (scatter)")
        fig.savefig(out_path + f'\{col}_scatter.png')
        plt.close(fig)

        #boxplot to look for outliers
        fig2 = plt.figure(figsize=(8, 4))
        sns.boxplot(x=df_outliers[col])
        plt.title(f"{col} (boxplot)")
        fig2.savefig(out_path + f'\{col}_boxplot.png')
        plt.close(fig2)


Z-scores : Describes the point's position relative to the mean of the group (column) in amount of standard deviations - here an outlier would be a point with a z-score of higher than abs(3). The algorithm is not considered as effective since it requires (assumes) a mean. However, potentially useful to know around how many points to expect as outliers during multidimensional analysis.

In [3]:
def zfunc(column, df_zscore):
    counter = 0
    for index, cell in df_zscore[column].items(): 
        if (not math.isnan(cell)) and (cell>3 or cell<-3):
            counter+=1
            df_zscore.loc[index, 'outlier']= 1
    log_message(path='/z_score_log.txt', msg = f'Number of outliers in column: {column} is: {counter}')

def outlier_method_z_scores(df_measures):

    df_zscore = stats.zscore(df_measures.copy())
    df_zscore['outlier'] = 0

    for col in df_measures.columns: 
        zfunc(col, df_zscore)

    counter = 0
    for cell in df_zscore['outlier']: 
        if cell == 1: counter += 1
    log_message(path='/z_score_log.txt', msg = f'Number of total outliers is: {counter}')


IQR : Measures the spread of the middle 50% of the data in a column - here an outlier would be a point that would fall out of the 50% range by +/- 1.5 times that range.

In [4]:
def iqr_func(column, df_iqr):
    q75, q25 = np.percentile(df_iqr[column], [75, 25])
    iqr = q75 - q25
    valid = iqr*2.5 
    counter = 0
    for index, cell in df_iqr[column].items():
        if  (not math.isnan(cell)) and (cell>q75+valid or cell<q25-valid): 
            df_iqr.loc[index, 'outlier'] = 1
            counter+=1
    log_message(path='/iqr_log.txt', msg = f'Number of outliers in column: {column} is: {counter}')

def outlier_method_iqr(df_measures):

    df_iqr = df_measures.copy()
    df_iqr['outlier'] = 0 

    for col in df_measures.columns: 
        iqr_func(col, df_iqr)

    counter = 0
    for cell in df_iqr['outlier']: 
        if cell ==1: counter += 1
    log_message(path='/iqr_log.txt', msg = f'Number of total outliers is: {counter}')

Isolation forest : Detects anomalies based on distance taking multiple parameters into consideration.

In [5]:
def outlier_method_isolation_forest(df_measures): 
    
    df_iso = df_measures.copy()

    iso_forest = IsolationForest(contamination=0.08, random_state=42) #for contamination came to the conclusion using the maximum amount of outliers found in the per column analysis before 
    iso_forest.fit(df_iso)
    df_iso['anomaly'] = iso_forest.predict(df_iso)

    counter = 0
    for index, row in df_iso.iterrows():  
            if row['anomaly']==-1: 
                counter +=1

    return df_iso

In [6]:
def remove_outliers(df_outliers, df_iso):
     
    #first adding the thermal_sensation column here to use in model later
    df_iso['thermal_sensation'] = df_outliers['thermal_sensation']

    #using isolation forest to handle outliers 
    df_final = df_iso[df_iso['anomaly'] != -1]

    #finding percentage of dataloss 
    loss = 100 - ((df_final['ta'].size/df_iso['ta'].size)*100)

    size_orig = df_iso['ta'].size
    size_final = df_final['ta'].size 
    log_message(msg = f"By removing outliers, with starting data size being: {size_orig} and new size being: {size_final} the data loss is: {loss}%", path = '/losses_log.txt')

    return df_final, loss


## Predictor

In [7]:
#RANDOM FOREST - haven't looked into improving yet

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

def random_forest(df_final):

    data = df_final.copy()

    X = data[['ta', 'rh', 'vel', 'met', 'clo', 'tr', 'season', 'age', 'pmv']].values
    y = data['thermal_sensation'].values.astype(int)

    # Shift -3..3 → 0..6 for classification
    y_class = y + 3

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_class, test_size=0.3, random_state=42
    )

    # Scale features (optional for RF)
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #hyperparameter tuning with GridSearchCV
    param_grid = {
        'n_estimators': [200, 400, 600],
        'max_depth': [10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    }

    rf = RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )

    best_rf = RandomForestClassifier(
      n_estimators=600,
      max_depth=20,
      min_samples_split=5,
      min_samples_leaf=2,
      max_features='sqrt',
      random_state=42,
      class_weight='balanced',
      n_jobs=-1
    )

    from sklearn.metrics import make_scorer

    def tolerance_accuracy(y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred) <= 1)

    tolerance_scorer = make_scorer(tolerance_accuracy)

    scoring = {
        'accuracy': 'accuracy',
        'f1_weighted': 'f1_weighted',
        'tolerance_acc': tolerance_scorer
    }

    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring= scoring, 
        refit='tolerance_acc',
        verbose=2
    )

    #best_rf.fit(X_train_scaled, y_train)
    grid_search.fit(X_train_scaled, y_train)
    best_rf = grid_search.best_estimator_
    #print("Best Random Forest parameters:", grid_search.best_params_)

    #do prediction
    y_pred_class = best_rf.predict(X_test_scaled)
    y_pred_final = y_pred_class - 3  # convert back to -3..3

    #accuracy
    exact_acc = accuracy_score(y_test - 3, y_pred_final)
    print(f"Exact accuracy: {exact_acc*100:.2f}%")

    # ±1 tolerance accuracy
    tolerance_acc = np.mean(np.abs(y_pred_final - (y_test - 3)) <= 1)
    print(f"±1 point tolerance accuracy: {tolerance_acc*100:.2f}%")

    # mean absolute error
    mae = mean_absolute_error(y_test - 3, y_pred_final)
    print(f"Mean Absolute Error: {mae:.3f}")

    # root mean squared error 
    rmse = mean_squared_error(y_test - 3, y_pred_final)
    print(f"Mean Absolute Error: {rmse:.3f}")

    # feature importance
    importances = best_rf.feature_importances_
    feature_names = ['ta', 'rh', 'vel', 'met', 'clo', 'tr', 'season', 'age', 'pmv']
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    print("Feature importances:\n", importance_df)

    results = pd.DataFrame(grid_search.cv_results_)
    print(results[['params', 'mean_test_accuracy', 'mean_test_f1_weighted', 'mean_test_tolerance_acc']])

ModuleNotFoundError: No module named 'numpy'

In [ ]:
#ORIGINAL MLP - change validation maybe

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

def linear_regression(df_final):
    
    #create data
    data = df_final.copy()
    X = data[['ta', 'rh', 'vel', 'met', 'clo', 'tr', 'season', 'age', 'pmv']].values
    y = data[['thermal_sensation']].values

    #separate into training and test datasets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 42)

    #scale data
    x_scaler = MinMaxScaler()
    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled = x_scaler.transform(X_test)

    y_scaler = MinMaxScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    y_test_scaled = y_scaler.transform(y_test)


    #create model
    model = Sequential()
    model.add(Dense(128, kernel_initializer= 'uniform', activation= 'relu', input_dim = 9))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(64, kernel_initializer= 'uniform', activation= 'relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(32, kernel_initializer= 'uniform', activation= 'relu'))
    model.add(Dense(16, kernel_initializer= 'uniform', activation= 'relu'))
    model.add(Dense(1, kernel_initializer= 'uniform', activation= 'linear'))

    model.summary()
    optimizer = Adam(learning_rate=1e-2)
    model.compile(optimizer= optimizer, loss = 'mse', metrics = ['mse', 'mae'])

    #fit model
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(X_train_scaled, y_train_scaled, batch_size= 128, epochs= 100, validation_split = 0.2, verbose = 1)

    #make predictions
    y_pred_scaled = model.predict(X_test_scaled)

    import math
    #errors on scaled
    rmse = mean_squared_error(y_pred_scaled, y_test_scaled)
    print("Root mean squares error for scaled: ",  math.sqrt(rmse))

    mae = mean_absolute_error(y_pred_scaled, y_test_scaled)
    print("Mean absolute error for scaled: ", mae)

    #inverse
    y_pred_unscaled = y_scaler.inverse_transform(y_pred_scaled)
    y_pred_rounded = np.round(y_pred_unscaled)
    y_pred_final = np.clip(y_pred_rounded, -3, 3)

    y_test_unscaled = y_scaler.inverse_transform(y_test_scaled)

    #errors on unscaled
    rmse = mean_squared_error(y_pred_final, y_test_unscaled)
    print("Root mean squares error for unscaled: ", math.sqrt(rmse))

    mae = mean_absolute_error(y_pred_final, y_test_unscaled)
    print("Mean absolute error for unscaled: ", mae)

    #accuracy
    accuracy = np.mean(abs(y_pred_final - y_test_unscaled) < mae)
    missed_one_point = np.mean(abs(y_pred_final - y_test_unscaled) < 1.5)
    print("Discrete Accuracy: ", accuracy)
    print("Percentage of misses by 1 point: ", missed_one_point)

    # TODO: rmse reflects bigger error when not neutral value - weight classes and do validation
    # also check why early stopping never stops early on all methods


In [ ]:
#NEW MLP - changing validation

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

def linear_regression_new(df_final):
    
    #create data
    data = df_final.copy()
    X = data[['ta', 'rh', 'vel', 'met', 'clo', 'tr', 'thermal_preference', 'season', 'age', 'pmv']].values
    y = data[['thermal_sensation']].values

    #separate into training and test datasets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 42)

    #scale data
    x_scaler = MinMaxScaler()
    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled = x_scaler.transform(X_test)

    y_scaler = MinMaxScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    y_test_scaled = y_scaler.transform(y_test)


    #create model
    model = Sequential()
    model.add(Dense(128, kernel_initializer= 'uniform', activation= 'relu', input_dim = 10))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(64, kernel_initializer= 'uniform', activation= 'relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(32, kernel_initializer= 'uniform', activation= 'relu'))
    model.add(Dense(16, kernel_initializer= 'uniform', activation= 'relu'))
    model.add(Dense(1, kernel_initializer= 'uniform', activation= 'linear'))

    model.summary()
    optimizer = Adam(learning_rate=1e-2)
    model.compile(optimizer= optimizer, loss = 'mse', metrics = ['mse', 'mae'])

    #fit model
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(X_train_scaled, y_train_scaled, batch_size= 128, epochs= 100, validation_split = 0.2, verbose = 1)

    #make predictions
    y_pred_scaled = model.predict(X_test_scaled)

    import math
    #errors on scaled
    rmse = mean_squared_error(y_pred_scaled, y_test_scaled)
    print("Root mean squares error for scaled: ",  math.sqrt(rmse))

    mae = mean_absolute_error(y_pred_scaled, y_test_scaled)
    print("Mean absolute error for scaled: ", mae)

    #inverse
    y_pred_unscaled = y_scaler.inverse_transform(y_pred_scaled)
    y_pred_rounded = np.round(y_pred_unscaled)
    y_pred_final = np.clip(y_pred_rounded, -3, 3)

    y_test_unscaled = y_scaler.inverse_transform(y_test_scaled)

    #errors on unscaled
    rmse = mean_squared_error(y_pred_final, y_test_unscaled)
    print("Root mean squares error for unscaled: ", math.sqrt(rmse))

    mae = mean_absolute_error(y_pred_final, y_test_unscaled)
    print("Mean absolute error for unscaled: ", mae)

    #accuracy
    accuracy = np.mean(abs(y_pred_final - y_test_unscaled) < mae)
    missed_one_point = np.mean(abs(y_pred_final - y_test_unscaled) < 1.5)
    print("Discrete Accuracy: ", accuracy)
    print("Percentage of misses by 1 point: ", missed_one_point)

    # TODO: rmse reflects bigger error when not neutral value - weight classes and do validation
    # also check why early stopping never stops early on all methods

In [ ]:
#CLASSIFICATION - needs tuning

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy

def classification(df_final):
    data = df_final.copy()

    X = data[['ta', 'rh', 'vel', 'met', 'clo', 'tr', 'thermal_preference', 'season', 'age', 'pmv']].values
    y = data[['thermal_sensation']].values.astype(int)

    #labels
    label_enc = LabelEncoder()
    y = label_enc.fit_transform(y)

    #split data
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )


    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
    )

    #scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    #model
    input_dim = X_train.shape[1]

    model = keras.Sequential([
        layers.Dense(128, activation="relu", kernel_initializer="he_normal", input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(64, activation="relu", kernel_initializer="he_normal"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation="relu", kernel_initializer="he_normal"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(7, activation="softmax")  # 7 classes (-3..+3)
    ])

    # compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", SparseTopKCategoricalAccuracy(k=2, name="top2_acc")]
    )

    #callbacks
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )

    checkpoint = keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_loss",
        save_best_only=True
    )

    #fit model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=[early_stop, reduce_lr, checkpoint],
        verbose=1
    )

    #evaluate
    test_loss, test_acc, test_top2 = model.evaluate(X_test, y_test, verbose=0)
    print(f"Final Test Accuracy: {test_acc*100:.2f}%")
    print(f"Final Test Top-2 Accuracy: {test_top2*100:.2f}%")

In [ ]:
def check_class_balance(df_final):
    df_final['thermal_sensation'] = (
        df_final['thermal_sensation']
        .round()                # round to nearest integer
        .clip(lower=-3, upper=3)  # ensure within [-3, 3]          
    )
    counts = df_final['thermal_sensation'].value_counts().sort_index()

    return df_final

##  Run file

In [ ]:
import pandas as pd 
import pathlib
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats as stats
import math
from IPython.display import display
import numpy as np 
from sklearn.ensemble import IsolationForest
import os

def log_message(path, msg, console=False):
    
    out_path = str(pathlib.Path().resolve() / "outputs")
    log_path = out_path + path
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")
    if console:
        print(msg)

def preprocessing(): 
    
    #make dataframe out of ASHRAE database saved file
    directory = "./database_files"
    with open(os.path.join(directory, "db_measurements_v2.1.0.csv")) as f:
        df = pd.read_csv(f) 

    #handle NaN values and choose relevant data columns
    df_outliers, df_measures, nan_loss = nan_handling(df)

    #handle outliers
    
    #plot data for better understanding
    data_plots(df_outliers)

    #do outlier detection on one feature at a time
    outlier_method_z_scores(df_measures)
    outlier_method_iqr(df_measures)

    #do outlier detection on multiple features and remove outliers
    df_iso = outlier_method_isolation_forest(df_measures)
    #df_final = df_measures.copy()
    #df_final['thermal_sensation'] = df_outliers['thermal_sensation']
    df_final, outlier_loss = remove_outliers(df_outliers, df_iso)

    df_final = check_class_balance(df_final)

    return df_final

def run_file(method, with_predict): 

    out_dir = pathlib.Path().resolve() / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    df_final = preprocessing()
    if (with_predict): 
        if (method == 'classification'):
            classification(df_final)
        elif (method == 'linear_regression'): 
            linear_regression(df_final)
        elif(method == 'random_forest'): 
            random_forest(df_final)
    
run_file('linear_regression', False)

C:\Users\MartyPickles\AppData\Local\Temp\ipykernel_4932\2971201087.py:26: DtypeWarning: Columns (5,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
C:\Users\MartyPickles\AppData\Local\Temp\ipykernel_4932\208971243.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['thermal_sensation'] = (
